# Entendimiento del Problema — Admisiones de Posgrado

**Autor:** Equipo SCO
**Fecha:** 2026-08-19

**Descripción:**
Análisis del Issue #1 "Descarga de los datos". Este notebook cubre el entendimiento
del problema, la carga reproducible del dataset RAW, una exploración inicial mínima
y la integración (aplazada) con el Feature Store de Hopsworks.


## 🎯 Entendimiento del problema (respuestas requeridas por el Issue #1)

1. **Objetivo del problema:** predecir la probabilidad de admisión de un candidato a un programa
   de posgrado a partir de su perfil académico, para que los estudiantes estimen sus chances.
2. **Tipo de problema:** regresión supervisada (la variable objetivo es continua en el rango 0–1).
3. **Variable objetivo:** `Chance of Admit` (0 a 1).
4. **Significado de cada variable:**
   - `GRE Score` (0–340): puntaje del GRE.
   - `TOEFL Score` (0–120): puntaje del TOEFL.
   - `University Rating` (0–5): rating de la universidad.
   - `SOP` (0–5): fortaleza del Statement of Purpose.
   - `LOR` (0–5): fortaleza de la Letter of Recommendation.
   - `CGPA` (0–10): GPA de pregrado.
   - `Research` (0/1): experiencia de investigación.
   - `Chance of Admit` (0–1): probabilidad de admisión (target).
5. **Fuente del dataset:** `data/01_raw/Admission_Predict.csv` (623 filas). Corresponde al dataset
   clásico *Graduate Admissions*; la fuente exacta y la licencia no están documentadas en el repo.
6. **Estructura del repositorio:** template cookiecutter (`data/`, `models/`, `notebooks/1-data…8-reports`,
   `scripts/`, `src/`, `tests/`). Ver `AGENTS.md`.
7. **Requisitos del curso para el Issue #1:** entendimiento del problema + integración mínima con
   Hopsworks como Feature Store (solo por exigencia explícita del Issue). Resto de producción fuera de alcance.
8. **Contenido del notebook:** las respuestas de esta lista, lectura reproducible del RAW, exploración
   mínima, conclusión del tipo de problema e integración Hopsworks documentada.
9. **Ubicación del RAW:** `data/01_raw/` (inmutable).
10. **Criterios de aceptación:** el notebook ejecuta de punta a punta; responde las 12 preguntas;
    lee el RAW de forma reproducible sin modificarlo; PR con revisión y CI verde.
11. **Archivos afectados:** este notebook (modificado) y `.env.example` (creado). Sin cambios en `src/`.
12. **Validaciones:** `ruff`, `pre-commit`, `pytest`, ejecución completa del notebook y revisión del `git diff`.


### Fuera de alcance en este Issue

No se realiza en el Issue #1: imputación de missing values, eliminación de outliers,
feature engineering, scaling, encoding, train/test split, baseline, entrenamiento de modelos,
model selection, MLflow, deployment, API, Docker ni pipelines FTI completos.


## 📚 Importar librerías


In [ ]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\samue\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [1]:
import os
from pathlib import Path

import pandas as pd

ModuleNotFoundError: No module named 'pandas'

## 💾 Carga reproducible del dataset RAW


In [2]:
def find_repo_root(start: Path) -> Path:
    """Localiza la raíz del repositorio subiendo hasta encontrar `pyproject.toml`."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


ROOT = find_repo_root(Path.cwd())
DATA_PATH = ROOT / "data" / "01_raw" / "Admission_Predict.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset en {DATA_PATH}.")

# El RAW se lee sin modificar; pandas convierte las celdas vacías a NaN.
raw_df = pd.read_csv(DATA_PATH)

# 'LOR ' y 'Chance of Admit ' traen espacio final: se limpia SOLO en memoria.
raw_df.columns = raw_df.columns.str.strip()

print(f"Dataset cargado desde: {DATA_PATH}")
raw_df.head()

NameError: name 'pd' is not defined

## 🔍 Exploración inicial mínima


In [ ]:
print(f"Shape (filas, columnas): {raw_df.shape}")
print(f"Columnas: {list(raw_df.columns)}")
print("\nTipos de datos:")
print(raw_df.dtypes)
print("\nValores faltantes por columna:")
print(raw_df.isna().sum())

## ✅ Conclusión sobre el tipo de problema

La variable objetivo `Chance of Admit` es continua (0–1), por lo que el problema es de **regresión**.
Los valores faltantes (7 de 8 columnas) se documentan aquí, pero se tratarán en etapas posteriores,
no en este Issue. El dataset RAW permanece inmutable.


## 🏪 Integración con Hopsworks (Feature Store)

El cuerpo del Issue #1 exige almacenar los datos históricos en un Feature Store (Hopsworks).

> ⚠️ **Aplazada.** El cliente `hopsworks` requiere `pandas<2.4` y `numpy<2.5`, incompatibles con
> `pandas 3.0.5` / `numpy 2.5.2` de este proyecto. El código siguiente queda listo para activarse
> cuando se resuelva ese conflicto de dependencias.

Pasos para habilitarla:

1. Decidir el downgrade de `pandas`/`numpy` (decisión arquitectónica pendiente).
2. `uv add hopsworks`.
3. Copiar `.env.example` a `.env` y completar `HOPSWORKS_API_KEY` y `HOPSWORKS_PROJECT`.
4. Re-ejecutar las celdas siguientes.


In [ ]:
try:
    import hopsworks
except ImportError:
    hopsworks = None


def load_env(env_path: Path) -> None:
    """Carga variables `KEY=VALUE` desde un archivo `.env` (sin dependencias extra)."""
    if not env_path.exists():
        return
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env(ROOT / ".env")

if hopsworks is None:
    print(
        "⚠️ El paquete `hopsworks` no está instalado. La integración con el "
        "Feature Store está aplazada por el conflicto pandas/numpy documentado "
        "en este notebook."
    )
else:
    api_key = os.environ.get("HOPSWORKS_API_KEY")
    project_name = os.environ.get("HOPSWORKS_PROJECT")

    if not api_key:
        raise RuntimeError(
            "HOPSWORKS_API_KEY no está definida. Copiá .env.example a .env y completá la key."
        )
    if not project_name:
        raise RuntimeError(
            "HOPSWORKS_PROJECT no está definida. Copiá .env.example a .env y completá el proyecto."
        )

    # hopsworks.login() lee HOPSWORKS_API_KEY / HOPSWORKS_PROJECT del entorno.
    project = hopsworks.login()
    fs = project.get_feature_store()

    # El dataset no tiene clave primaria: se agrega un id sintético (no es imputación ni FE).
    data_for_fs = raw_df.copy()
    data_for_fs.insert(0, "application_id", range(len(data_for_fs)))

    feature_group_name = "graduate_admissions"
    feature_group_version = 1

    fg = fs.get_or_create_feature_group(
        name=feature_group_name,
        version=feature_group_version,
        primary_key=["application_id"],
        description="Datos históricos de admisiones de posgrado (POC - Issue #1)",
    )
    # Carga de los datos históricos (sin event_time por ahora).
    fg.insert(data_for_fs)

    # Evidencia verificable: lectura de vuelta y validación de la cantidad de filas.
    stored_df = fg.read()
    assert len(stored_df) == len(raw_df), (
        f"Filas esperadas {len(raw_df)}, obtenidas {len(stored_df)}."
    )

    print("✅ Validación OK: los datos se almacenaron correctamente.")
    print(f"Feature group: {fg.name} v{fg.version}")
    print(f"Filas: {len(stored_df)} — Columnas: {list(stored_df.columns)}")
    stored_df.head()

## 📊 Conclusiones y próximos pasos

- El problema es de regresión; el target es `Chance of Admit`.
- El dataset RAW (623 filas) presenta valores faltantes en 7 de 8 columnas; se tratarán en etapas posteriores.
- La integración con Hopsworks queda aplazada por el conflicto `pandas`/`numpy` (ver sección anterior).
- Próximos pasos: exploración/EDA, tratamiento de missing values, feature engineering y baseline.


## 📖 Referencias

- Dataset *Graduate Admissions* (fuente original a confirmar): [Kaggle - Mohan S. Acharya](https://www.kaggle.com/datasets/mohansacharya/graduate-admissions)
- Ejemplo de backfill de features con Hopsworks: [air-quality-fti](https://github.com/JoseRZapata/air-quality-fti/blob/main/notebooks/1-data/1_air_quality_feature_backfill.ipynb)
- Documentación de Hopsworks: <https://docs.hopsworks.ai>
